# UNet Segmentation Pipeline

Main orchestration notebook for the U-Net segmentation pipeline.
Run each stage independently as needed:

| Stage | When to run |
|---|---|
| 1 – Preprocessing | Once per dataset (or when data changes) |
| 2 – Tuning | Optional; skip if using default hyperparameters |
| 3 – Training | Each time you want to train a new model |
| 4 – Evaluation | After training, to measure final performance |

## Setup

In [ ]:
# Configuration — choose which config file to use.
# Each config file contains all ML hyperparameters (learning rate, batch size,
# model architecture, etc.). Swap to a different config by changing this import.
import config.configUnetxAE as configuration

# Pipeline stage modules
import preprocessing
import tuning
import training
import evaluation

# Load and validate the configuration.
# validate() checks that all required parameters are present and sensible.
config = configuration.Configuration().validate()
print("Configuration loaded:", config.modality)

## Stage 1: Preprocessing

Loads raw geospatial images, normalizes pixel values, creates image chips, and splits into train/val/test sets.

**Run when:** You have new or modified input data. Skip if you have already preprocessed and chips are saved to disk.

In [ ]:
preprocessing.preprocess_all(config)

## Stage 2: Hyperparameter Tuning (Optional)

Automatically searches for the best hyperparameters (learning rate, dropout, etc.) using Optuna.

**Run when:** Starting fresh or when default parameters are not yielding good results. This is computationally expensive — skip if you have a known good config.

In [ ]:
# Uncomment to run hyperparameter tuning:
# best = tuning.tune_UNet(config)
# print("Best hyperparameters:", best)

## Stage 3: Training

Trains the U-Net model. Running multiple iterations (the loop below) averages results across random seeds, giving more stable performance estimates.

### TensorBoard
Run the cell below to open TensorBoard inline. It will show live loss curves and image predictions as training progresses.

> **Note:** If TensorBoard does not appear inline, open a terminal and run:
> ```
> venv-bubble/Scripts/tensorboard --logdir <config.logs_dir>
> ```
> then open `http://localhost:6006` in your browser.

In [ ]:
# Launch TensorBoard inline.
# Logs are written to config.logs_dir during training.
%load_ext tensorboard
%tensorboard --logdir {config.logs_dir}

In [ ]:
# Train the model. Each iteration starts from fresh random weights.
# Multiple iterations reduce variance in the final metrics.
for i in range(10):
    print(f"\n =========== Starting training iteration {i+1}/10 ===========\n")
    training.train_UNet(config)

## Stage 4: Evaluation

Evaluates the trained model on the held-out test set. This is the only unbiased measure of real-world performance, because:
- Training data was used to learn weights
- Validation data was used to select the best checkpoint
- **Test data is completely unseen** — metrics here reflect true generalization

In [ ]:
evaluation.evaluate_unet(config)